# Tutorial 18: Text Knowledge Embeddings

This tutorial demonstrates how to embed biological entities as **text
descriptions** using embpy's `TextResolver` and `embed_description()`.

### What this tutorial covers

1. **TextResolver standalone** -- fetch descriptions for genes, proteins,
   and molecules from 6 knowledge sources
2. **embed_description()** -- resolve + embed in a single call
3. **Compare text vs sequence embeddings** -- do they agree?
4. **Custom text embedding** -- embed arbitrary user-provided text

### Knowledge sources

| Source | Entities | What it returns |
|---|---|---|
| MyGene.info | Genes | NCBI summary, gene name, pathways |
| NCBI Gene | Genes | Curated gene description |
| Ensembl | Genes | Gene biotype and description |
| UniProt | Proteins | Function, subcellular location, disease |
| Wikipedia | All | Article extract |
| PubChem | Molecules | Compound description |

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import torch
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import embpy.pl as epl

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)
%matplotlib inline

---
## 1. TextResolver standalone

Fetch descriptions for a gene from all 6 sources.

In [ ]:
from embpy.resources.text_resolver import TextResolver

tr = TextResolver(organism='human')
descs = tr.get_gene_description('TP53', sources='all')

for source, text in descs.items():
    preview = text[:200] + '...' if len(text) > 200 else text
    print(f'\n[{source.upper()}] ({len(text)} chars)')
    print(f'  {preview}')

In [ ]:
combined = tr.get_combined_description('TP53', entity_type='gene')
print(f'Combined description ({len(combined)} chars):\n')
print(combined[:500])

### Protein descriptions

In [ ]:
prot_descs = tr.get_protein_description('TP53')
for source, text in prot_descs.items():
    print(f'[{source.upper()}] {text[:150]}...' if text else f'[{source.upper()}] (empty)')

### Molecule descriptions

In [ ]:
mol_descs = tr.get_molecule_description('aspirin')
for source, text in mol_descs.items():
    print(f'[{source.upper()}] {text[:200]}...' if text else f'[{source.upper()}] (empty)')

---
## 2. embed_description()

Resolve a biological entity to text and embed it in a single call.

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device='auto')

genes = ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS']
drugs = ['aspirin', 'ibuprofen', 'dexamethasone']

text_embs = {}
for entity in genes + drugs:
    try:
        emb = embedder.embed_description(entity, model='minilm_l6_v2')
        text_embs[entity] = emb
        print(f'  {entity:15s}  dim={emb.shape[0]}')
    except Exception as e:
        print(f'  {entity:15s}  FAILED: {e}')

In [ ]:
entities = [e for e in genes + drugs if e in text_embs]
entity_types = ['gene'] * len([g for g in genes if g in text_embs]) + \
               ['molecule'] * len([d for d in drugs if d in text_embs])

adata = ad.AnnData(
    obs=pd.DataFrame({'entity': entities, 'type': entity_types},
                      index=pd.Index(entities)),
)
adata.obsm['X_text'] = np.stack([text_embs[e] for e in entities]).astype(np.float32)

epl.plot_similarity_heatmap(
    adata=adata, obsm_key='X_text', metric='cosine',
    labels=entities, title='Text embedding cosine similarity',
    annot=True, fmt='.2f',
)
plt.show()

---
## 3. Compare text vs sequence embeddings

Embed the same genes via protein sequence (ESM-2) and text description
(MiniLM), then compare the similarity structures.

In [ ]:
prot_embs = {}
for gene in genes:
    try:
        emb = embedder.embed_protein(gene, model='esm2_650M', pooling_strategy='mean')
        prot_embs[gene] = emb
        print(f'  {gene:10s}  ESM-2 dim={emb.shape[0]}')
    except Exception as e:
        print(f'  {gene:10s}  FAILED: {e}')

common = [g for g in genes if g in text_embs and g in prot_embs]
if len(common) >= 3:
    adata_cmp = ad.AnnData(
        obs=pd.DataFrame({'gene': common}, index=pd.Index(common)),
    )
    adata_cmp.obsm['X_text'] = np.stack([text_embs[g] for g in common]).astype(np.float32)
    adata_cmp.obsm['X_esm2'] = np.stack([prot_embs[g] for g in common]).astype(np.float32)

    epl.cross_embedding_correlation(
        adata_cmp, obsm_key_a='X_text', obsm_key_b='X_esm2',
        method='pearson',
    )
    plt.show()

---
## 4. Custom text embedding

`embed_text()` accepts any raw text string -- useful for custom
descriptions, hypotheses, or experimental notes.

In [ ]:
custom_texts = [
    'TP53 is a tumor suppressor that responds to DNA damage.',
    'BRCA1 repairs double-strand DNA breaks via homologous recombination.',
    'Aspirin inhibits cyclooxygenase enzymes COX-1 and COX-2.',
]

for text in custom_texts:
    emb = embedder.embed_text(text, model='minilm_l6_v2')
    print(f'  "{text[:60]}..."  dim={emb.shape[0]}')

---
## Summary

| Method | Input | Output |
|---|---|---|
| `TextResolver.get_gene_description(gene)` | Gene symbol | Dict of source -> text |
| `TextResolver.get_protein_description(prot)` | Gene/UniProt ID | Dict of source -> text |
| `TextResolver.get_molecule_description(mol)` | Drug name/SMILES | Dict of source -> text |
| `TextResolver.get_combined_description(id)` | Any | Single combined text string |
| `embedder.embed_description(id, model)` | Any | Embedding vector |
| `embedder.embed_text(text, model)` | Raw text | Embedding vector |